In [3]:
import time
import re
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    WebDriverException,
)

# =========================
# 0. 설정
# =========================

# ✅ 최종 병합본 CSV (너가 만든 babitalk_final_merged.csv)
INPUT_CSV = "babitalk_final_merged.csv"

# ✅ 진단 결과 저장 파일
OUTPUT_CSV = "babitalk_dom_diagnostics.csv"

MAX_URLS = None   # None이면 전부, 숫자 넣으면 상위 N개만 테스트용


# =========================
# 1. 셀렉터 정의 (크롤러와 동일하게)
# =========================

TREATMENT_NAME_SELECTOR = "h1.text-label-common_5"
HOSPITAL_NAME_SELECTOR = "p.text-label-common_4.cursor-pointer"
HOSPITAL_SECTION_SELECTOR = "#hospital"

ORIGINAL_PRICE_SELECTORS = [
    "h6.text-label-skinTreatment_2",
    "h6.text-label-plasticsurgery_2",
    "h2 span",
]

SALE_PRICE_SELECTOR = "h2.flex.items-center.gap-\\[2px\\]"
CONTENT_BODY_SELECTOR = "#content-body"


# =========================
# 2. 유틸 함수
# =========================

def init_driver():
    chromedriver_autoinstaller.install()
    options = Options()
    options.add_argument("--start-maximized")
    # options.add_argument("--headless")  # 필요하면 활성화
    driver = webdriver.Chrome(options=options)
    return driver


def element_exists(driver, selector):
    """해당 셀렉터 요소가 하나라도 있으면 True"""
    try:
        els = driver.find_elements(By.CSS_SELECTOR, selector)
        return len(els) > 0
    except Exception:
        return False


def any_price_exists(driver):
    """정가/할인가 관련 요소가 하나라도 있으면 True"""
    # 정가 후보들
    for sel in ORIGINAL_PRICE_SELECTORS:
        if element_exists(driver, sel):
            return True

    # 할인가
    if element_exists(driver, SALE_PRICE_SELECTOR):
        return True

    # h2 중 '원' 들어간 텍스트 있으면 가격으로 판단
    try:
        h2s = driver.find_elements(By.CSS_SELECTOR, "h2")
        for h in h2s:
            t = h.text.strip()
            if "원" in t:
                return True
    except Exception:
        pass

    return False


def any_percent_h2_exists(driver):
    """h2 중 '%'가 포함된 텍스트가 있으면 True"""
    try:
        h2s = driver.find_elements(By.CSS_SELECTOR, "h2")
        for h in h2s:
            t = h.text.strip()
            if "%" in t:
                return True
    except Exception:
        pass
    return False


# =========================
# 3. 단일 URL 진단
# =========================

def diagnose_url(driver, url, wait_time=10):
    info = {
        "treatment_url": url,
        "has_content_body": False,
        "has_treatment_h1": False,
        "has_hospital_name_p": False,
        "has_price_block": False,
        "has_any_percent_h2": False,
        "has_hospital_section": False,
        "alert_shown": False,
        "error_message": None,
        "page_type_guess": None,
        "reason": None,
    }

    try:
        driver.get(url)

        # content-body 대기(있으면)
        try:
            WebDriverWait(driver, wait_time).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, CONTENT_BODY_SELECTOR))
            )
        except TimeoutException:
            # content-body 자체가 없는 경우도 진단해야 해서 여기서 바로 return 안 함
            pass

        time.sleep(0.5)

        # 기본 구조 체크
        info["has_content_body"] = element_exists(driver, CONTENT_BODY_SELECTOR)
        info["has_treatment_h1"] = element_exists(driver, TREATMENT_NAME_SELECTOR)
        info["has_hospital_name_p"] = element_exists(driver, HOSPITAL_NAME_SELECTOR)
        info["has_hospital_section"] = element_exists(driver, HOSPITAL_SECTION_SELECTOR)
        info["has_price_block"] = any_price_exists(driver)
        info["has_any_percent_h2"] = any_percent_h2_exists(driver)

        # 페이지 타입/원인 추정 로직
        info = classify_page_type(info)

    except WebDriverException as e:
        info["error_message"] = str(e)
        info["page_type_guess"] = "webdriver_error"
        info["reason"] = "브라우저/네트워크 오류로 페이지 로드 실패"

    return info


def classify_page_type(info):
    """
    DOM 스캔 결과를 바탕으로 페이지 타입 & 수집 실패 이유 추정
    """

    has_body = info["has_content_body"]
    has_name = info["has_treatment_h1"]
    has_hosp = info["has_hospital_name_p"]
    has_price = info["has_price_block"]
    has_percent = info["has_any_percent_h2"]
    has_hosp_sec = info["has_hospital_section"]

    # 1) content-body 자체가 없으면: 리다이렉트/삭제/에러 페이지 가능성
    if not has_body:
        info["page_type_guess"] = "no_content_body"
        info["reason"] = "content-body 영역이 없어 시술 상세 템플릿이 아님 (리다이렉트/삭제/특수 페이지 가능성)"
        return info

    # 2) body는 있는데 시술명 / 병원명 / 가격 모두 없음 → 비시술형 이벤트
    if has_body and (not has_name) and (not has_hosp) and (not has_price):
        info["page_type_guess"] = "non_medical_event"
        info["reason"] = "시술명·병원명·가격 블록이 모두 없어 프로모션/공지형 이벤트일 가능성이 큼"
        return info

    # 3) 시술명은 있는데 가격/병원 없음 → 정보 비노출형 이벤트
    if has_name and (not has_price) and (not has_hosp):
        info["page_type_guess"] = "name_only_event"
        info["reason"] = "시술명은 있지만 가격/병원 정보가 없어 상담유도형 또는 비공개 가격 이벤트로 추정"
        return info

    # 4) 병원 섹션(#hospital)은 있는데 병원명 p가 없음 → 다른 템플릿/구조
    if has_hosp_sec and (not has_hosp):
        info["page_type_guess"] = "hospital_section_without_name"
        info["reason"] = "병원 섹션은 있으나 병원명 텍스트 요소가 없어 템플릿 구조가 다르거나 컨텐츠 비노출"
        return info

    # 5) 가격은 있는데 시술명 없음 → 헤더 구조 변경/셀렉터 미스매치
    if has_price and (not has_name):
        info["page_type_guess"] = "price_without_name"
        info["reason"] = "가격 요소는 있으나 시술명 h1을 찾지 못해 셀렉터 미스매치 가능성이 큼"
        return info

    # 6) 할인율(%)만 있고 나머지 없음 → 특수 가격 카드
    if has_percent and (not has_price) and (not has_name):
        info["page_type_guess"] = "percent_only_card"
        info["reason"] = "할인율로 추정되는 % 텍스트는 있으나 시술명/가격 구조가 달라 일반 템플릿이 아님"
        return info

    # 7) 그 외: unknown
    info["page_type_guess"] = "unknown_pattern"
    info["reason"] = "DOM 구조가 예상 패턴과 다르지만 일부 요소는 존재함. 셀렉터 추가 분석 필요"

    return info


# =========================
# 4. 메인 – 결측 URL 자동 진단
# =========================

def main():
    # 1) 최종 병합본 로드
    df = pd.read_csv(INPUT_CSV)

    # 2) "시술명 / 가격 / 병원명" 모두 결측인 URL만 필터링
    mask_failed = df["treatment_name"].isna() & df["original_price_text"].isna() & df["hospital_name"].isna()
    failed_urls = df.loc[mask_failed, "treatment_url"].dropna().unique().tolist()

    if MAX_URLS is not None:
        failed_urls = failed_urls[:MAX_URLS]

    print(f"진단 대상 URL 수: {len(failed_urls)}")

    driver = init_driver()
    results = []

    try:
        for idx, url in enumerate(failed_urls, start=1):
            print(f"[{idx}/{len(failed_urls)}] 진단 중: {url}")
            info = diagnose_url(driver, url)
            results.append(info)
            time.sleep(0.5)
    finally:
        driver.quit()

    diag_df = pd.DataFrame(results)
    diag_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ DOM 진단 완료 → {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


진단 대상 URL 수: 429
[1/429] 진단 중: https://web.babitalk.com/events/58479
[2/429] 진단 중: https://web.babitalk.com/events/63455
[3/429] 진단 중: https://web.babitalk.com/events/64900
[4/429] 진단 중: https://web.babitalk.com/events/69541
[5/429] 진단 중: https://web.babitalk.com/events/71835
[6/429] 진단 중: https://web.babitalk.com/events/59413
[7/429] 진단 중: https://web.babitalk.com/events/57023
[8/429] 진단 중: https://web.babitalk.com/events/60024
[9/429] 진단 중: https://web.babitalk.com/events/70365
[10/429] 진단 중: https://web.babitalk.com/events/69420
[11/429] 진단 중: https://web.babitalk.com/events/67811
[12/429] 진단 중: https://web.babitalk.com/events/67152
[13/429] 진단 중: https://web.babitalk.com/events/63560
[14/429] 진단 중: https://web.babitalk.com/events/68333
[15/429] 진단 중: https://web.babitalk.com/events/71349
[16/429] 진단 중: https://web.babitalk.com/events/68334
[17/429] 진단 중: https://web.babitalk.com/events/68388
[18/429] 진단 중: https://web.babitalk.com/events/68849
[19/429] 진단 중: https://web.babitalk.co